# Làm sạch dữ liệu Product Data

Notebook xử lý data từ `product_data.json` → `product_data_cleaned.json`

In [1]:
import json
import re
from collections import Counter

with open("product_data.json", "r", encoding="utf-8") as f:
    products = json.load(f)

print(f"Tổng sản phẩm: {len(products)}")

Tổng sản phẩm: 1463


## Bước 1: Xóa sản phẩm trùng lặp (`product_id`)

In [2]:
seen = set()
unique_products = []
for p in products:
    if p["product_id"] not in seen:
        seen.add(p["product_id"])
        unique_products.append(p)

print(f"Trước: {len(products)} → Sau dedup: {len(unique_products)}")
print(f"Đã xóa: {len(products) - len(unique_products)} bản trùng")

Trước: 1463 → Sau dedup: 1312
Đã xóa: 151 bản trùng


## Bước 2: Chuẩn hóa text (bỏ underscore, ký tự đặc biệt)

In [3]:
def clean_text(text):
    if not text:
        return ""
    # Bỏ underscore nối từ
    text = text.replace("_", " ")
    # Bỏ ký tự đặc biệt đầu (ღ, ★, ...)
    text = re.sub(r'^[^\w\s]+\s*', '', text)
    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test
print(clean_text("ღ THÔNG_TIN SẢN_PHẨM sản_phẩm"))
print(clean_text("Đầm sơ_mi xanh mint NGADO , đầm công_sở"))

ღ THÔNG TIN SẢN PHẨM sản phẩm
Đầm sơ mi xanh mint NGADO , đầm công sở


In [4]:
TEXT_FIELDS = ["title", "short_description", "description"]

for p in unique_products:
    for field in TEXT_FIELDS:
        p[field] = clean_text(p.get(field, ""))
    # Breadcrumbs
    p["breadcrumbs"] = [clean_text(b) for b in p.get("breadcrumbs", [])]
    # Origin
    p["origin"] = clean_text(p.get("origin", ""))

# Kiểm tra
print("Title:", unique_products[0]["title"])
print("Breadcrumbs:", unique_products[0]["breadcrumbs"])

Title: Đầm sơ mi xanh mint NGADO , đầm công sở , đầm dự tiệc
Breadcrumbs: ['Thời trang nữ', 'Đầm nữ', 'Đầm suông', 'Đầm sơ mi xanh mint NGADO , đầm công sở , đầm dự tiệc']


## Bước 3: Loại bỏ boilerplate Tiki trong description

In [5]:
BOILERPLATE_PATTERN = re.compile(
    r'Giá sản phẩm Tiki bao gồm thuế.*$',
    re.DOTALL
)

boilerplate_count = 0
for p in unique_products:
    before = p["description"]
    p["description"] = BOILERPLATE_PATTERN.sub('', p["description"]).strip()
    # Bỏ dấu . thừa cuối chuỗi
    p["description"] = re.sub(r'[\s.]+$', '', p["description"])
    if before != p["description"]:
        boilerplate_count += 1

print(f"Boilerplate removed: {boilerplate_count}/{len(unique_products)}")

Boilerplate removed: 1312/1312


## Bước 4: Deduplicate image URLs

In [6]:
def deduplicate_images(product):
    urls = product.get("image_urls", [])
    high_res = []
    seen_hashes = set()
    for url in urls:
        match = re.search(r'/([a-f0-9]{32})\.\w+$', url)
        if match:
            img_hash = match.group(1)
            if img_hash not in seen_hashes:
                seen_hashes.add(img_hash)
                # Ưu tiên bản w1200
                w1200_url = re.sub(r'cache/200x280/', 'cache/w1200/', url)
                high_res.append(w1200_url)
        else:
            high_res.append(url)
    product["image_urls"] = high_res

avg_before = sum(len(p.get("image_urls", [])) for p in unique_products) / len(unique_products)

for p in unique_products:
    deduplicate_images(p)

avg_after = sum(len(p["image_urls"]) for p in unique_products) / len(unique_products)
print(f"Avg image URLs: {avg_before:.1f} → {avg_after:.1f}")

Avg image URLs: 38.0 → 10.6


## Bước 5: Chuẩn hóa trường origin

In [7]:
ORIGIN_MAP = {
    "china": "Trung Quốc",
    "vietnam": "Việt Nam",
    "hong kong": "Hồng Kông",
    "korea": "Hàn Quốc",
    "japan": "Nhật Bản",
    "thailand": "Thái Lan",
}

def clean_origin(origin):
    if not origin:
        return "Không rõ"

    origin_lower = origin.lower().strip().strip(",").strip()

    # Map tiếng Anh → tiếng Việt
    for eng, viet in ORIGIN_MAP.items():
        origin_lower = origin_lower.replace(eng, viet.lower())

    # Nếu chứa nhiều nước → lấy nước đầu tiên
    parts = re.split(r'[,\s]+', origin)
    countries = [p.strip() for p in parts if len(p.strip()) > 2 and p.strip() != "..."]

    if not countries:
        return "Không rõ"

    return ", ".join(countries[:2]) if len(countries) > 1 else countries[0]

for p in unique_products:
    p["origin"] = clean_origin(p["origin"])

# Thống kê
origin_counts = Counter(p["origin"] for p in unique_products)
print("Top origins:")
for origin, count in origin_counts.most_common(10):
    print(f"  {origin}: {count}")

Top origins:
  Việt, Nam: 1043
  Trung, Quốc: 157
  Không rõ: 48
  China, Vietnam: 19
  Thái, Lan: 9
  Hàn, Quốc: 8
  Hong, Kong: 7
  Trung, Quôc: 5
  Tây, Ban: 4
  HKTQ: 3


## Bước 6: Loại bỏ short_description trùng description

In [8]:
removed_count = 0
for p in unique_products:
    desc = p["description"]
    short = p["short_description"]
    if desc and short and desc.startswith(short[:50]):
        p["short_description"] = ""
        removed_count += 1

print(f"short_description trùng đã xóa: {removed_count}/{len(unique_products)}")

short_description trùng đã xóa: 1097/1312


## Bước 7: Export data sạch

In [9]:
cleaned = []
for p in unique_products:
    cleaned.append({
        "product_id": p["product_id"],
        "title": p["title"],
        "description": p["description"] or p["short_description"],
        "category": p["breadcrumbs"][1] if len(p["breadcrumbs"]) > 1 else "",
        "subcategory": p["breadcrumbs"][2] if len(p["breadcrumbs"]) > 2 else "",
        "brand": p.get("brand", {}).get("name", ""),
        "origin": p["origin"],
        "options": p.get("configurable_options", []),
        "image_urls": p["image_urls"],
    })

with open("product_data_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(cleaned, f, ensure_ascii=False, indent=2)

print(f"Exported {len(cleaned)} cleaned products → product_data_cleaned.json")

Exported 1312 cleaned products → product_data_cleaned.json


## Thống kê tổng quan data sau khi clean

In [10]:
print("=" * 50)
print("THỐNG KÊ SAU KHI CLEAN")
print("=" * 50)
print(f"Tổng products: {len(cleaned)}")
print(f"Có description: {sum(1 for p in cleaned if p['description'])}")
print(f"Có brand: {sum(1 for p in cleaned if p['brand'])}")
print(f"Có origin: {sum(1 for p in cleaned if p['origin'] != 'Không rõ')}")
print(f"Có options: {sum(1 for p in cleaned if p['options'])}")
print(f"Avg image URLs: {sum(len(p['image_urls']) for p in cleaned) / len(cleaned):.1f}")

# Category distribution
cat_counts = Counter(p["category"] for p in cleaned if p["category"])
print(f"\nSố categories: {len(cat_counts)}")
print("Top 10 categories:")
for cat, count in cat_counts.most_common(10):
    print(f"  {cat}: {count}")

THỐNG KÊ SAU KHI CLEAN
Tổng products: 1312
Có description: 1309
Có brand: 1312
Có origin: 1264
Có options: 1204
Avg image URLs: 10.6

Số categories: 15
Top 10 categories:
  Áo liền quần - trang phục: 90
  Chân váy: 90
  Quần nữ: 90
  Áo vest - Áo khoác nữ: 90
  Đồ đôi - Đồ gia đình: 90
  Thời trang nữ trung niên: 90
  Đồ lót nữ: 90
  Đồ ngủ - Đồ mặc nữ: 90
  Trang phục bơi nữ: 90
  Áo sơ mi nữ: 90
